In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import KBinsDiscretizer
import warnings
from sklearn.decomposition import PCA
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.feature_selection import VarianceThreshold, SelectKBest, chi2
from sklearn.preprocessing import PowerTransformer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold

In [11]:
train = pd.read_csv("train.csv")
X = train.drop(columns=["class"])
y = train["class"]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


from sklearn.model_selection import StratifiedKFold, cross_validate

from sklearn.model_selection import cross_validate

def evaluate(model, X, y):

    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring="accuracy",
        return_train_score=True
    )

    train_mean = scores["train_score"].mean()
    val_mean = scores["test_score"].mean()
    val_std = scores["test_score"].std()

    return train_mean, val_mean, val_std


Los principales parametros son tres:
- n_estimators — cuántos stumps entrenar en secuencia. Más estimadores = más correcciones, pero a partir de cierto punto el modelo converge y añadir más no mejora. Típicamente se explora entre 50 y 300.
- learning_rate — cuánto contribuye cada modelo nuevo a la corrección. Hay un trade-off clásico con n_estimators: learning rate bajo necesita más estimadores para converger, learning rate alto converge antes pero puede sobreajustar. Valores típicos: 0.01 – 1.0.
- max_depth del estimador base — aunque técnicamente es un parámetro del árbol, en AdaBoost es crucial. Lo normal es max_depth=1 (stump), pero a veces se prueba con 2 o 3 para darle más capacidad al modelo base.

### MODELO BASE

Primero creamos un modelo base con  (150 stumps, learning_rate=0.2) para tener un modelo de referencia y poder mejorarlo.

In [12]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

ada_base = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=150,
    learning_rate=0.2,
    random_state=42
)

train, cv_mean, cv_std = evaluate(ada_base, X, y)
print(f"AdaBoost baseline → Train: {train:.4f} | CV: {cv_mean:.4f} ± {cv_std:.4f}")

AdaBoost baseline → Train: 0.8020 | CV: 0.7640 ± 0.0233


 Primero vamos a buscar el max_depth óptimo del estimador base, que es el parámetro más impactante.

Es el parámetro más impactante porque afecta la capacidad base del ensemble entero. Si cada árbol aprende poco, necesitas muchos para llegar a un buen resultado. Si cada árbol aprende demasiado, el boosting pierde su ventaja y aparece overfitting.

In [9]:
for depth in [1, 2, 3, 4, 5]:
    ada = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=depth),
        n_estimators=150,
        learning_rate=0.2,
        random_state=42
    )
    train, cv_mean, cv_std = evaluate(ada, X, y)
    print(f"max_depth={depth} → Train: {train:.4f} | CV: {cv_mean:.4f} ± {cv_std:.4f}")

max_depth=1 → Train: 0.8020 | CV: 0.7640 ± 0.0233
max_depth=2 → Train: 0.9357 | CV: 0.7930 ± 0.0129
max_depth=3 → Train: 0.9995 | CV: 0.7920 ± 0.0214
max_depth=4 → Train: 1.0000 | CV: 0.7960 ± 0.0097
max_depth=5 → Train: 1.0000 | CV: 0.8120 ± 0.0186


El problema es que a partir de max_depth=3 el árbol ya memoriza todo el train (1.0000) y las mejoras en CV son pequeñas. Con max_depth=5 el CV es el mejor pero el gap es enorme.
Yo me quedaría con max_depth=2 — es donde el CV da el salto más grande (+0.029 sobre depth=1) con un overfitting todavía controlado (gap=0.14 vs 0.20+ de los demás).

Después de fijar el max_depth óptimo, el siguiente paso es ajustar n_estimators y learning_rate juntos, porque están acoplados.

In [14]:
import numpy as np

lrs = [0.5, 0.2, 0.1, 0.05, 0.01]
ns = [100, 200, 300, 500, 1000]

print(f"{'lr':<8} {'n_est':<8} {'Train':>8} {'CV':>8} {'Gap':>8}")
print("-" * 45)

for lr in lrs:
    for n in ns:
        ada = AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=2),
            n_estimators=n,
            learning_rate=lr,
            random_state=42
        )
        train, cv_mean, cv_std = evaluate(ada, X, y)
        print(f"lr={lr:<6} n={n:<6} {train:>8.4f} {cv_mean:>8.4f} {train-cv_mean:>8.4f}")
    print()

lr       n_est       Train       CV      Gap
---------------------------------------------
lr=0.5    n=100      0.9645   0.7910   0.1735
lr=0.5    n=200      0.9798   0.7880   0.1918
lr=0.5    n=300      0.9835   0.7880   0.1955
lr=0.5    n=500      0.9855   0.7920   0.1935
lr=0.5    n=1000     0.9875   0.8000   0.1875

lr=0.2    n=100      0.9205   0.7910   0.1295
lr=0.2    n=200      0.9527   0.7930   0.1598
lr=0.2    n=300      0.9623   0.7960   0.1663
lr=0.2    n=500      0.9718   0.7900   0.1818
lr=0.2    n=1000     0.9755   0.7920   0.1835

lr=0.1    n=100      0.8785   0.7830   0.0955
lr=0.1    n=200      0.9107   0.7850   0.1258
lr=0.1    n=300      0.9323   0.7930   0.1392
lr=0.1    n=500      0.9553   0.7990   0.1562
lr=0.1    n=1000     0.9653   0.7990   0.1663

lr=0.05   n=100      0.8420   0.7660   0.0760
lr=0.05   n=200      0.8780   0.7780   0.1000
lr=0.05   n=300      0.8968   0.7830   0.1138
lr=0.05   n=500      0.9190   0.7860   0.1330
lr=0.05   n=1000     0.9567   0.


lr=0.5 n=1000 tiene CV=0.8000, apenas 0.001 más, pero con gap de 0.1875 — no merece la pena.

Con lo que tenemos, me quedo con lr=0.Con lo que tenemos, me quedo con lr=0.1 y n=500:

CV = 0.7990 — el más alto de lr=0.1
Gap = 0.1562 — razonable, no se dispara como con lr=0.5
lr=0.1 es el punto donde el CV sube con n sin que el overfitting se desboque

In [15]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import make_pipeline

k_values = [10, 20, 30, 40, 50, 60]

print(f"{'k':<8} {'Train':>8} {'CV':>8} {'Gap':>8}")
print("-" * 35)

for k in k_values:
    pipe = make_pipeline(
        SelectKBest(f_classif, k=k),
        AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=2),
            n_estimators=500,
            learning_rate=0.1,
            random_state=42
        )
    )
    train, cv_mean, cv_std = evaluate(pipe, X, y)
    print(f"k={k:<6} {train:>8.4f} {cv_mean:>8.4f} {train-cv_mean:>8.4f}")

k           Train       CV      Gap
-----------------------------------
k=10       0.8285   0.7750   0.0535
k=20       0.8678   0.7950   0.0727
k=30       0.8918   0.8020   0.0898
k=40       0.9052   0.7960   0.1092
k=50       0.9113   0.7990   0.1122
k=60       0.9185   0.8050   0.1135


La mejor combinación fue lr=0.1 con n=500, que ofrecía el mejor CV sin disparar el overfitting como hacía lr=0.5. Por último se añadió SelectKBest para filtrar las features más relevantes, probando valores de k entre 10 y 60 y afinando entre 25 y 35. El valor óptimo fue k=30, que redujo el ruido de entrada y mejoró tanto el CV como el gap. El modelo final pasó de un CV de 0.7640 en el baseline a 0.8020, con un gap de solo 0.09 — muy por debajo del overfitting que mostraban configuraciones más agresivas.

Probamos  a ver si con un criterio de SelectKbest cambia

In [16]:
from sklearn.feature_selection import mutual_info_classif

pipe_mi = make_pipeline(
    SelectKBest(mutual_info_classif, k=30),
    AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=2),
        n_estimators=500,
        learning_rate=0.1,
        random_state=42
    )
)
train, cv_mean, cv_std = evaluate(pipe_mi, X, y)
print(f"mutual_info k=30 → Train: {train:.4f} | CV: {cv_mean:.4f} ± {cv_std:.4f}")

mutual_info k=30 → Train: 0.9013 | CV: 0.7920 ± 0.0181


f_classif (ANOVA) mide si la media de cada feature es significativamente distinta entre clases. Es lineal — detecta bien features que separan clases de forma directa y clara.
mutual_info_classif mide dependencias más complejas y no lineales entre cada feature y la clase. En teoría es más potente, pero en la práctica con pocos datos o mucho ruido puede seleccionar features que parecen informativas pero no generalizan bien.

### MODELO FINAL

In [17]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline

modelo_final_ada = make_pipeline(
    SelectKBest(f_classif, k=30),
    AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=2),
        n_estimators=500,
        learning_rate=0.1,
        random_state=42
    )
)

train, cv_mean, cv_std = evaluate(modelo_final_ada, X, y)
print(f"Modelo Final AdaBoost → Train: {train:.4f} | CV: {cv_mean:.4f} ± {cv_std:.4f}")

Modelo Final AdaBoost → Train: 0.8918 | CV: 0.8020 ± 0.0378


max_depth=2 es un techo
Viste que max_depth=3 disparaba el overfitting sin mejorar el CV. Eso significa que el modelo con max_depth=2 ya está en su límite de capacidad — no puede aprender más sin memorizar. El bagging no tiene ese problema porque compensa la complejidad con la agregación.
El feature selection ayudó pero no fue suficiente



k=30 mejoró el CV de 0.7990 a 0.8020, pero el ruido restante de las otras features en el bagging se maneja mejor porque los árboles profundos pueden ignorar features irrelevantes de forma natural.